In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import GPT2Tokenizer
from transformers import GPT2LMHeadModel,Trainer,TrainingArguments
from transformers import DataCollatorForLanguageModeling
from datasets import Dataset

In [36]:
# loading data
with open('event_chains.txt') as f:
    sequences = [line for line in f]
#print(sequences)

In [38]:
# transformation to the dataframe
df = pd.DataFrame({"sequence":sequences})
# print(df)

In [40]:
# split the training set and the validation set
train_df,val_df = train_test_split(df,test_size=0.2,random_state=42)

In [42]:
# initializing the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"pad_token":"[PAD]"})    # to make every token has the same length,fill the empty with [PAD]

1

In [43]:
# sending the sequences to tokenizer,setting up a series of parameters,dealing with 2 kinds of data set
def encode_sequences(sequence):
    encoded = tokenizer(sequences,padding="max_length",max_length=16,truncation=True,return_tensors="pt")
    return encoded
train_encoded = encode_sequences(train_df["sequence"].tolist())
val_encoded = encode_sequences(val_df["sequence"].tolist())
train_set = Dataset.from_dict(train_encoded)
val_set = Dataset.from_dict(val_encoded)

In [44]:
# loading pretrained model
model = GPT2LMHeadModel.from_pretrained("distilgpt2")
#definging the arguments
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)
training_args = TrainingArguments(remove_unused_columns=False,learning_rate=3e-5,per_device_train_batch_size=2,per_device_eval_batch_size=2)
#defining the trainer
trainer = Trainer(model=model,args=training_args,train_dataset=train_set,eval_dataset=val_set,data_collator=data_collator)

In [47]:
# train!
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


TrainOutput(global_step=57, training_loss=3.4626055265727795, metrics={'train_runtime': 12.2289, 'train_samples_per_second': 9.322, 'train_steps_per_second': 4.661, 'total_flos': 465434836992.0, 'train_loss': 3.4626055265727795, 'epoch': 3.0})

NameError: name 'GPT2LMHeadModel' is not defined